<a href="https://www.kaggle.com/code/vijay20213/f1-pit-stop-prediction-xgb-lgb-cb?scriptVersionId=322415638" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Library Imports and Configs

In [1]:
!pip uninstall -y pytabkit torch torchvision torchaudio -q

!pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu121 -q

!pip install pytabkit -q

!pip install optbinning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 81.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 90.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 71.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 54.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 102.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 34.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 15.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import os
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
import optuna
import xgboost as xgb 
from optbinning import OptimalBinning
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import plotly.graph_objects as go
import math
import random
import warnings
from tqdm import tqdm
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)

def seed_everything(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

PyTorch  version: 2.5.1+cu121
True
Tesla P100-PCIE-16GB


In [3]:
ROOT_PATH = "/kaggle/input/competitions/playground-series-s6e5"

train_df = pd.read_csv(os.path.join(ROOT_PATH, "train.csv"))
test_df = pd.read_csv(os.path.join(ROOT_PATH, "test.csv"))
sub_df = pd.read_csv(os.path.join(ROOT_PATH, "sample_submission.csv"))

train_df.shape, test_df.shape 

((439140, 16), (188165, 15))

In [4]:
F_ENGG = True
MAX_BIN = 25
MIN_BIN = 0.06
SHOW_WOE = False
RMLP_OPTUNA = False
XGB_OPTUNA = False
LGB_OPTUNA = False
CAT_OPTUNA = False
TEST_BEFORE_SUB = False
DEACTIVATE_XGB_LGB_CAT = False

## Feature Engineering
- Domain Knowledge
- Correlation after feature engineering

In [5]:
t_col = "PitNextLap"
ID = "id"
c_cols = test_df.drop([ID], axis='columns').select_dtypes(include='object').columns.to_list()
n_cols = test_df.drop([ID], axis='columns').select_dtypes(exclude='object').columns.to_list()
print("Categorical columns: ", c_cols)

category_map = {}
important_combos = [
    ('Race', 'Compound'),
    ('Race', 'Year'),
]

def engineer_race_features(df, fit=False):
    # Arithmetic interaction
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')

    # Categorize string cats
    for col in c_cols:
        if fit:
            codes, uniques = df[col].factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = df[col].map(code_map).fillna(-1).astype('int32')
        df[col] = codes
        df[col] = df[col].astype('category')
    
    # Categorize numericals
    for col in n_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']: 
        cat_name = f"{col}_cat_" if col in n_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype('category')
    
    # Count encoding
    for col in c_cols + ['Year_cat_', 'PitStop_cat_']:
        count_name = f"_{col}_count" if col in c_cols else f"_{col[:-1]}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].astype(object).map(count_map).fillna(0).astype('int32')

    # Discretize numericals
    bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype('category')

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype('category')   

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names


print(f"Applying feature engineering on train and test ...")
train_df_eng, new_cat_cols, new_num_cols, combo_names = engineer_race_features(train_df, fit=True)
test_df_eng, new_cat_cols, new_num_cols, combo_names = engineer_race_features(test_df, fit=False)


c_cols += new_cat_cols
n_cols += new_num_cols

Categorical columns:  ['Driver', 'Compound', 'Race']
Applying feature engineering on train and test ...


## Data Processing
- Encoding - (categorical data) (Label Encoding and OneHotEncoding)
- Normalization - (numerical data)

## Target Encoding - OOF

In [6]:
from sklearn.model_selection import StratifiedKFold

encoded_cat_cols = c_cols

train_df_scaled_v1 = train_df_eng.copy()
test_df_scaled_v1 = test_df_eng.copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in encoded_cat_cols:
    train_df_scaled_v1[f'{col}_mean'] = 0.0
    train_df_scaled_v1[f'{col}_std'] = 0.0

X, y = train_df_scaled_v1.drop([t_col], axis='columns'), train_df_scaled_v1[t_col]

for train_idx, val_idx in skf.split(X, y):
    train_fold = train_df_scaled_v1.iloc[train_idx]
    val_fold = train_df_scaled_v1.iloc[val_idx]

    for col in encoded_cat_cols:
        stats = train_fold.groupby(col)[t_col].agg(['mean', 'std'])
        val_fold = val_fold.merge(stats, on=col, how='left')

        train_df_scaled_v1.loc[val_idx, f'{col}_mean'] = val_fold['mean'].values
        train_df_scaled_v1.loc[val_idx, f'{col}_std'] = val_fold['std'].values

        val_fold.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    train_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = train_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

for col in encoded_cat_cols:
    stats = train_df_scaled_v1.groupby(col)[t_col].agg(['mean', 'std'])
    test_df_scaled_v1 = test_df_scaled_v1.merge(stats, on=col, how='left')

    test_df_scaled_v1[f'{col}_mean'] = test_df_scaled_v1['mean']
    test_df_scaled_v1[f'{col}_std'] = test_df_scaled_v1['std']

    test_df_scaled_v1.drop(['mean', 'std'], axis=1, inplace=True)

for col in encoded_cat_cols:
    test_df_scaled_v1[[f'{col}_mean', f'{col}_std']] = test_df_scaled_v1[
        [f'{col}_mean', f'{col}_std']
    ].fillna(0)

## WoE Transformation
- WoE
- IV

In [7]:
class OptimalBinner:

    def __init__(
        self, c_cols: list, n_cols: list, max_n_bins: int = 5, min_bin_size: float = 0.05):
        self.c_cols = c_cols
        self.n_cols = n_cols
        self.max_n_bins = max_n_bins
        self.min_bin_size = min_bin_size
        self.binners = {}

    def fit(self, X, y, verbos=True):
        for col in X.columns:
            if verbos:
                print(f"Fitting: {col}")
            if col in self.c_cols:
                optb = OptimalBinning(
                    name=col,
                    dtype="categorical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )

            else:
                optb = OptimalBinning(
                    name=col,
                    dtype="numerical",
                    max_n_bins=self.max_n_bins,
                    min_bin_size=self.min_bin_size
                )
            # Fit binning
            optb.fit(X[col], y)
            self.binners[col] = optb
        return self

    def transform(self, X, metric="woe"):
        X_transformed = pd.DataFrame(index=X.index)
        for col in X.columns:
            optb = self.binners[col]
            X_transformed[col] = optb.transform(
                X[col],
                metric=metric,
                metric_missing="empirical",
                metric_special="empirical"
            )
        return X_transformed

    def get_iv_summary(self):
        iv_data = []
        for col, optb in self.binners.items():
            iv = optb.binning_table.build()["IV"].sum()
            iv_data.append({
                "feature": col,
                "iv": iv
            })
        return (
            pd.DataFrame(iv_data)
            .sort_values("iv", ascending=False)
            .reset_index(drop=True)
        )

    def get_binning_table(self, col):
        return self.binners[col].binning_table.build()


class LGB_XGB_CAT_Model:

    def __init__(self, lgb_params=None, xgb_params=None, cb_params=None):

        self.lgb_model = None
        self.xgb_model = None
        self.cb_model = None

        self.lgb_params = lgb_params
        self.xgb_params = xgb_params
        self.cb_params = cb_params

    def train(self, X_train, y_train):

        if type(self.lgb_params) != type(None):
            print("Training LGB model...")
            self.lgb_model = lgb.LGBMClassifier(
                **self.lgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.xgb_params) != type(None):
            print("Training XGB model...")
            self.xgb_model = xgb.XGBClassifier(
                **self.xgb_params
            ).fit(
                X_train,
                y_train
            )

        if type(self.cb_params) != type(None):
            print("Training CatBoost model...")
            self.cb_model = cb.CatBoostClassifier(
                **self.cb_params
            ).fit(
                X_train,
                y_train
            )
        print("Models trained successfully!")

    def predict_proba(self, X):
        probs = {}
        if type(self.lgb_params) != type(None):
            probs['lgb'] = (
                self.lgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.xgb_params) != type(None):
            probs['xgb'] = (
                self.xgb_model
                .predict_proba(X)[:, 1]
            )

        if type(self.cb_params) != type(None):
            probs['cb'] = (
                self.cb_model
                .predict_proba(X)[:, 1]
            )
        return probs

    def roc_auc_score(self, X_test, y_test):
        roc_auc_dict = {}
        if type(self.lgb_params) != type(None):
            roc_auc_dict['lgb'] = roc_auc_score(y_test, self.lgb_model.predict_proba(X_test)[:, 1])

        if type(self.xgb_params) != type(None):
            roc_auc_dict['xgb'] = roc_auc_score(y_test, self.xgb_model.predict_proba(X_test)[:, 1])

        if type(self.cb_params) != type(None):
            roc_auc_dict['cb'] = roc_auc_score(y_test, self.cb_model.predict_proba(X_test)[:, 1])

        return roc_auc_dict

    def cross_validation(self, X, y, groups, n_splits=5):
        gkf = GroupKFold(
            n_splits=n_splits
        )
        cv_scores = {
            "lgb": [],
            "xgb": [],
            "cb": []
        }

        for train_idx, valid_idx in gkf.split(X, y, groups=groups):
            X_train_cv = X.iloc[train_idx]
            X_valid_cv = X.iloc[valid_idx]

            y_train_cv = y.iloc[train_idx]
            y_valid_cv = y.iloc[valid_idx]

            if type(self.lgb_params) != type(None):

                model_lgb = lgb.LGBMClassifier(**self.lgb_params)
                model_lgb.fit(X_train_cv, y_train_cv)
                y_prob_lgb = model_lgb.predict_proba(X_valid_cv)[:, 1]
                auc_lgb = roc_auc_score(y_valid_cv, y_prob_lgb)
                cv_scores["lgb"].append(auc_lgb)

            if type(self.xgb_params) != type(None):
                model_xgb = xgb.XGBClassifier(**self.xgb_params)
                model_xgb.fit(X_train_cv, y_train_cv)
                y_prob_xgb = model_xgb.predict_proba(X_valid_cv)[:, 1]
                auc_xgb = roc_auc_score(y_valid_cv, y_prob_xgb)
                cv_scores["xgb"].append(auc_xgb)

            if type(self.cb_params) != type(None):
                model_cb = cb.CatBoostClassifier(**self.cb_params)
                model_cb.fit(X_train_cv, y_train_cv, verbose=False)
                y_prob_cb = model_cb.predict_proba(X_valid_cv)[:, 1]
                auc_cb = roc_auc_score(y_valid_cv, y_prob_cb)
                cv_scores["cb"].append(auc_cb)

        final_scores = {}

        if len(cv_scores["lgb"]) > 0:
            final_scores["lgb"] = np.mean(cv_scores["lgb"])

        if len(cv_scores["xgb"]) > 0:
            final_scores["xgb"] = np.mean(cv_scores["xgb"])

        if len(cv_scores["cb"]) > 0:
            final_scores["cb"] = np.mean(cv_scores["cb"])

        return final_scores


X, y = train_df_scaled_v1.drop(columns=[t_col, ID], axis='columns'), train_df_scaled_v1[t_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,  stratify=y)
X_oot = test_df_scaled_v1.drop(columns=[ID], axis='columns').copy()

MAX_BIN, MIN_BIN = 25, 0.06

if MAX_BIN == None or MIN_BIN == None:
    max_n_bins = [5, 10, 15, 20, 25]
    min_bin_size = [0.001, 0.05, 0.06, 0.07]
    best_bin_range = {}
    best_roc_auc = 0.0

    print(f"Searching for best max_bins and min_bins combination...")
    for max_bin in max_n_bins:
        for min_bin in min_bin_size:
            print(f"Fitting with max_bin = {max_bin} and min_bin = {min_bin}...")
            bin_obj = OptimalBinner(
                c_cols=c_cols,
                n_cols=n_cols,
                max_n_bins=max_bin,
                min_bin_size=min_bin
            )

            bin_obj.fit(X_train, y_train, verbos=False)
            X_test_woe = bin_obj.transform(X_test, metric="woe")

            X_test_train, X_test_test, y_test_train, y_test_test = train_test_split(X_test_woe, y_test, test_size=0.3, random_state=42,  stratify=y_test)
            
            model_base = LGB_XGB_CAT_Model(xgb_params={})
            model_base.train(X_test_train, y_test_train)
            
            auc = model_base.roc_auc_score(X_test_test, y_test_test).values()
            avg_auc = sum(list(auc)) / len(auc)
            
            if best_roc_auc < avg_auc:
                best_roc_auc = avg_auc
                best_bin_range['max_bin'] = max_bin
                best_bin_range['min_bin'] = min_bin
            print(f"BIN RANGE : {max_bin, min_bin} | BEST BIN RANGE : {best_bin_range} | AUC : {avg_auc} | BEST AUC : {best_roc_auc}")

    MAX_BIN = best_bin_range['max_bin']
    MIN_BIN = best_bin_range['min_bin']

bin_obj = OptimalBinner(
    c_cols=c_cols,
    n_cols=n_cols,
    max_n_bins=MAX_BIN,
    min_bin_size=MIN_BIN
)

bin_obj.fit(X_train, y_train, verbos=False)

X_train_woe = bin_obj.transform(X_train, metric="woe")
X_test_woe = bin_obj.transform(X_test, metric="woe")
X_oot_woe = bin_obj.transform(X_oot, metric="woe")

print("X_train_woe shape:", X_train_woe.shape)
print("X_test_woe shape:", X_test_woe.shape)
print("X_oot_woe shape:", X_oot_woe.shape)

X_train_woe shape: (307398, 81)
X_test_woe shape: (131742, 81)
X_oot_woe shape: (188165, 81)


## Ordinal Encoder

In [8]:
# from sklearn.preprocessing import OrdinalEncoder
# import pandas as pd


# def ordinal_encode_features(
#     train_df: pd.DataFrame,
#     test_df: pd.DataFrame,
#     categorical_columns: list,
#     unknown_value: int = -1
# ):

#     # Copy dataframes
#     train_encoded = train_df.copy()
#     test_encoded = test_df.copy()

#     # Initialize encoder
#     ord_encoder = OrdinalEncoder(
#         handle_unknown="use_encoded_value",
#         unknown_value=unknown_value
#     )

#     # Fit on train and transform both
#     train_transformed = ord_encoder.fit_transform(
#         train_encoded[categorical_columns]
#     )

#     test_transformed = ord_encoder.transform(
#         test_encoded[categorical_columns]
#     )

#     # Create encoded column names
#     encoded_columns = [
#         col.replace("engg", "en") if "engg" in col else f"{col}_en"
#         for col in categorical_columns
#     ]

#     # Add encoded columns
#     train_encoded[encoded_columns] = train_transformed
#     test_encoded[encoded_columns] = test_transformed

#     # Remove original categorical columns
#     train_encoded.drop(columns=categorical_columns, inplace=True)
#     test_encoded.drop(columns=categorical_columns, inplace=True)

#     return train_encoded, test_encoded, ord_encoder

# X_ord, X_oot, _ = ordinal_encode_features(X, X_oot, c_cols)
# X_train, X_test, y_train, y_test = train_test_split(X_ord, y, test_size=0.3, random_state=42,  stratify=y)

In [9]:
# X_train_full = pd.concat([X_train_woe.rename(columns=lambda x: x + '_woe'), X_train], axis='columns')
# X_test_full = pd.concat([X_test_woe.rename(columns=lambda x: x + '_woe'), X_test], axis='columns')
# X_oot_full = pd.concat([X_oot_woe.rename(columns=lambda x: x + '_woe'), X_oot], axis='columns')

X_train_full = X_train
X_test_full = X_test
X_oot_full = X_oot

cat_cols = X_train_full.select_dtypes(include="category").columns

for col in cat_cols:
    X_train_full[col] = X_train_full[col].astype("int32")
    X_test_full[col] = X_test_full[col].astype("int32")
    X_oot_full[col] = X_oot_full[col].astype("int32")
    
X_final = pd.concat([X_train_full, X_test_full], axis='rows')
y_final = pd.concat([y_train, y_test], axis='rows')

X_train_full.shape, X_test_full.shape, X_oot_full.shape, X_final.shape, y_final.shape

((307398, 81), (131742, 81), (188165, 81), (439140, 81), (439140,))

## Model Training

In [10]:
def submission(model, test_data, file_name: str):
    pred_data = test_data.sort_index().copy()
    predictions = model.predict_proba(pred_data)[:, 1]
    if 'id' in pred_data.columns:
        ids = pred_data['id'].values
    else:
        ids = range(439140, 439140 + len(pred_data))

    sub_df = pd.DataFrame({
        'id': ids,
        'PitNextLap': predictions
    })

    sub_df.to_csv(file_name, index=False)
    print(f"Submission saved to: {file_name}")

In [11]:
count_neg = (y == 0).sum()
count_pos = (y == 1).sum()
scale_weight = count_neg / count_pos

# best_xgb_params = {
#     "objective": "binary:logistic",
#     "eval_metric": "auc",
#     "tree_method": "hist",

#     "n_estimators": 1844,
#     "learning_rate": 0.031608629117770966,
#     "max_depth": 8,
#     "min_child_weight": 4.372586656731096,
#     "subsample": 0.9631759623756463,
#     "colsample_bytree": 0.6509888828205632,
#     "gamma": 0.06992616873406238,
#     "reg_alpha": 0.0021813118475736935,
#     "reg_lambda": 41.64055084302312,
#     "scale_pos_weight": 2.725253225646067,

#     "random_state": 42,
#     "n_jobs": -1,
# }

best_xgb_params = {'n_estimators': 1387, 'learning_rate': 0.019396567559317207, 'max_depth': 12, 'min_child_weight': 3.5703128223751985, 'subsample': 0.9853307875901292, 'colsample_bytree': 0.4241677044102965, 'gamma': 1.3572322515033948, 'reg_alpha': 0.5637466103278904, 'reg_lambda': 41.27172380308245, 'scale_pos_weight': 2.3921907070602235, 'max_bin': 503}

xgb_model = XGBClassifier(
    **best_xgb_params
)

if not DEACTIVATE_XGB_LGB_CAT:
    if TEST_BEFORE_SUB:
        print("Validation Started ...")
        xgb_model.fit(X_train_full, y_train)
        y_prob_test_xgb = xgb_model.predict_proba(X_test_full)[:, 1]
        y_prob_train_xgb = xgb_model.predict_proba(X_train_full)[:, 1]
        
        print("ROC AUC Score - XGBoost")
        print(f"Testing:  {roc_auc_score(y_test, y_prob_test_xgb):.4f}")
        print(f"Training: {roc_auc_score(y_train, y_prob_train_xgb):.4f}")
    else:
        print("Train and Submission Started ...")
        xgb_model.fit(X_final, y_final)
        submission(xgb_model, X_oot_full, "XGB_Submission_V2.csv")

Train and Submission Started ...
Submission saved to: XGB_Submission_V2.csv


In [12]:
# lgb_best_params = {
#     "objective": "binary",
#     "metric": "auc",
#     "boosting_type": "gbdt",
#     "n_estimators": 1844,
#     "learning_rate": 0.031608629117770966,
#     "max_depth": 8,
#     "num_leaves": 255,
#     "min_child_weight": 4.372586656731096,
#     "subsample": 0.9631759623756463,
#     "colsample_bytree": 0.6509888828205632,
#     "reg_alpha": 0.0021813118475736935,
#     "reg_lambda": 41.64055084302312,
#     "scale_pos_weight": 2.725253225646067,
#     "verbosity": -1,
#     "random_state": 42,
#     "n_jobs": -1,
# }

lgb_best_params = {'n_estimators': 3384, 'learning_rate': 0.012233674973369394, 'num_leaves': 113, 'max_depth': 10, 'min_child_samples': 20, 'subsample': 0.9017148817821318, 'subsample_freq': 4, 'colsample_bytree': 0.7435811034511604, 'reg_alpha': 0.14437423433547297, 'reg_lambda': 0.13226606027443225, 'scale_pos_weight': 1.4438279510690053, 'max_bin': 170, 'min_gain_to_split': 0.9486893475961762}

lgb_model = LGBMClassifier(
   **lgb_best_params
)

if not DEACTIVATE_XGB_LGB_CAT:
    if TEST_BEFORE_SUB:
        lgb_model.fit(X_train_full, y_train)
        
        y_prob_test_lgb = lgb_model.predict_proba(X_test_full)[:, 1]
        y_prob_train_lgb = lgb_model.predict_proba(X_train_full)[:, 1]
        
        print("ROC AUC Score - LightGBM")
        print(f"Testing:  {roc_auc_score(y_test, y_prob_test_lgb):.4f}")
        print(f"Training: {roc_auc_score(y_train, y_prob_train_lgb):.4f}")
    else:
        print("Train and Submission Started ...")
        lgb_model.fit(X_final, y_final )
        submission(lgb_model, X_oot_full, "LGB_Submission_V2.csv")

Train and Submission Started ...
[LightGBM] [Warning] min_gain_to_split is set=0.9486893475961762, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.9486893475961762
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_gain_to_split is set=0.9486893475961762, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.9486893475961762
[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.231177 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7691
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 81
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGB

In [13]:
# cat_params = {
#     "loss_function": "Logloss",
#     "eval_metric": "AUC",
#     "iterations": 1844,
#     "learning_rate": 0.031608629117770966,
#     "depth": 8,
#     "min_data_in_leaf": 4,
#     "subsample": 0.9631759623756463,
#     "rsm": 0.6509888828205632,
#     "l2_leaf_reg": 41.64055084302312,
#     "scale_pos_weight": 2.725253225646067,
#     "random_seed": 42,
#     "verbose": 200,
# }

cat_params = {'iterations': 1635, 'learning_rate': 0.07354570098271265, 'depth': 8, 'bootstrap_type': 'Bayesian', 'l2_leaf_reg': 71.51279901739346, 'random_strength': 3.7417561846953302, 'scale_pos_weight': 1.5762947731022652, 'border_count': 231}

cat_model = cb.CatBoostClassifier(
    **cat_params,
    verbose = 200
) 

if not DEACTIVATE_XGB_LGB_CAT:
    if TEST_BEFORE_SUB:
        cat_model.fit(X_train_full, y_train)
        
        y_prob_test_cb = cat_model.predict_proba(X_test_full)[:, 1]
        y_prob_train_cb = cat_model.predict_proba(X_train_full)[:, 1]

        print("ROC AUC Score - LightGBM")
        print(f"Testing:  {roc_auc_score(y_test, y_prob_test_cb):.4f}")
        print(f"Training: {roc_auc_score(y_train, y_prob_train_cb):.4f}")
    else:
        cat_model.fit(X_final, y_final )
        submission(cat_model, X_oot_full, "CAT_Submission_V2.csv")

0:	learn: 0.6163793	total: 207ms	remaining: 5m 37s
200:	learn: 0.2632043	total: 25.8s	remaining: 3m 4s
400:	learn: 0.2495366	total: 50.4s	remaining: 2m 35s
600:	learn: 0.2419619	total: 1m 14s	remaining: 2m 8s
800:	learn: 0.2360946	total: 1m 39s	remaining: 1m 43s
1000:	learn: 0.2307985	total: 2m 3s	remaining: 1m 18s
1200:	learn: 0.2258195	total: 2m 28s	remaining: 53.5s
1400:	learn: 0.2209414	total: 2m 52s	remaining: 28.8s
1600:	learn: 0.2163608	total: 3m 16s	remaining: 4.17s
1634:	learn: 0.2156257	total: 3m 20s	remaining: 0us
Submission saved to: CAT_Submission_V2.csv


In [14]:
# CONFIG = {
#     "n_ens": 16,                 # renamed from n_ens
#     "embedding_size": 6,
#     "max_one_hot_cat_size": 4,        # renamed from onehot_thresh
#     "hidden_sizes": [256, 256, 256],  # renamed from hidden_dims
#     "p_drop": 0.05,                   # renamed from dropout
#     "p_drop_sched": "expm4t",
#     "act":'silu',                   # renamed from activation
#     "add_front_scale": True,
#     "plr_hidden_1": 20,               # renamed from pbld_hidden_dim
#     "plr_hidden_2": 4,                # pbld_out_dim - 1
#     "plr_sigma": 5.0,                 # renamed from pbld_freq_scale
#     "plr_act_name": 'prelu',              # renamed from pbld_activation
#     "plr_lr_factor": 0.093,           # renamed from pbld_lr_factor
#     "lr": 0.008,
#     "mom": 0.9,
#     "sq_mom": 0.98,
#     "lr_sched": "flat_cos",
#     "first_layer_lr_factor": 1.0,
#     "scale_lr_factor": 10.0,          # renamed from lr_scale_mult
#     "bias_lr_factor": 0.1,            # renamed from lr_bias_mult
#     "wd": 0.005,                      # renamed from weight_decay
#     "bias_wd_factor": 0.5,            # renamed from wd_bias_mult
#     "ls_eps": 0.04,
#     "ls_eps_sched": "cos",
#     "tfms": ["median_center", "robust_scale", "smooth_clip"],
#     "n_epochs": 4,                    # renamed from epochs
#     "batch_size": 256,                # renamed from train_bs
#     "predict_batch_size": 10240,      # renamed from eval_bs
#     "verbosity": 2,
#     "use_early_stopping": False,
#     "early_stopping_additive_patience": 10,
#     "early_stopping_multiplicative_patience": 1,
#     "device": "cuda",
#     "random_state": 42,
# }

CONFIG = {'n_ens': 16, 'embedding_size': 4, 'max_one_hot_cat_size': 3, 'hidden_sizes': [512, 256], 'p_drop': 0.14115374739891437, 'p_drop_sched': 'expm4t', 'act': 'silu', 'plr_hidden_1': 23, 'plr_hidden_2': 6, 'plr_sigma': 1.5879630047507403, 'plr_act_name': 'prelu', 'plr_lr_factor': 0.07107711950204332, 'lr': 0.009984752276554902, 'mom': 0.9786035058094887, 'sq_mom': 0.9965891797166353, 'lr_sched': 'constant', 'first_layer_lr_factor': 0.8291920495308914, 'scale_lr_factor': 16.49017531644613, 'bias_lr_factor': 0.8059603012070959, 'wd': 0.00019448169367212214, 'bias_wd_factor': 0.23122172473730493, 'ls_eps': 0.05957049626307287, 'ls_eps_sched': 'constant', 'n_epochs': 16, 'batch_size': 256}

# --- Fold split ---
FOLDS = 5
SEED = 42
TE = True

In [15]:
from pytabkit import RealMLP_TD_Classifier

rmlp_model = RealMLP_TD_Classifier(**CONFIG)

if TEST_BEFORE_SUB:
    rmlp_model.fit(
        X_train_full, y_train,
        X_test_full, y_test,
        cat_col_names=c_cols,
    )
    # --- RealMLP Evaluation ---
    y_prob_test_rmlp = rmlp_model.predict_proba(X_test_full)[:, 1]
    y_prob_train_rmlp = rmlp_model.predict_proba(X_train_full)[:, 1]
    
    print("ROC AUC Score - Real_MLP")
    print(f"Testing:  {roc_auc_score(y_test, y_prob_test_rmlp):.4f}")
    print(f"Training: {roc_auc_score(y_train, y_prob_train_rmlp):.4f}")

else:
    rmlp_model.fit(
        X_final, y_final
    )
    submission(rmlp_model, X_oot_full, "Real_MLP_Submission_V2.csv")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=16` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Submission saved to: Real_MLP_Submission_V2.csv


## Blending

In [16]:
def weighted_probability_blend(scores, predictions, normalize_weights=True):
    weights = np.array(scores, dtype=np.float64)
    if normalize_weights:
        weights = weights / weights.sum()
        
    blended_pred = np.zeros(len(predictions[0]), dtype=np.float64)
    
    for weight, pred in zip(weights, predictions):
        blended_pred += weight * np.array(pred)

    return blended_pred

blend_pred = weighted_probability_blend(
    scores=[0.95082, 0.95083, 0.95035, 0.95088, 0.95116],
    predictions=[
        pd.read_csv("/kaggle/working/XGB_Submission_V2.csv")['PitNextLap'].to_list(),
        pd.read_csv("/kaggle/working/LGB_Submission_V2.csv")['PitNextLap'].to_list(),
        pd.read_csv("/kaggle/working/CAT_Submission_V2.csv")['PitNextLap'].to_list(),
        pd.read_csv("/kaggle/working/Real_MLP_Submission_V2.csv")['PitNextLap'].to_list(),
        pd.read_csv("/kaggle/input/datasets/vijay20213/blended-predictions-f1-pit-stop-prediction/Blended_XGB_LGB_New_FE.csv")['PitNextLap'].to_list()
    ]
)

IDS = list(range(439140, 439140 + len(blend_pred)))
blended_sub = pd.DataFrame({"id":IDS, "PitNextLap":blend_pred})
blended_sub.to_csv("submission.csv", index=False)
blended_sub.head()

,id,PitNextLap
0,439140,0.012141
1,439141,0.019324
2,439142,0.012258
3,439143,0.268357
4,439144,0.881363


## RealMLP - Optuna

In [17]:
%%time
def objective(trial):
    config = {
        "n_ens": trial.suggest_categorical("n_ens", [4, 8, 16]),
        "embedding_size": trial.suggest_int("embedding_size", 4, 16),
        "max_one_hot_cat_size": trial.suggest_int("max_one_hot_cat_size", 2, 16),
        "hidden_sizes": trial.suggest_categorical("hidden_sizes", [[128, 128],[256, 128],[256, 256],[256, 256, 128],[256, 256, 256],[512, 256]]),
        "p_drop": trial.suggest_float("p_drop", 0.0, 0.3),
        "p_drop_sched": trial.suggest_categorical("p_drop_sched", ["constant", "expm4t"]),
        "act": trial.suggest_categorical("act", ["relu", "silu", "mish"]),
        "add_front_scale": True,

        "plr_hidden_1": trial.suggest_int("plr_hidden_1", 8, 64),
        "plr_hidden_2": trial.suggest_int("plr_hidden_2", 2, 16),
        "plr_sigma": trial.suggest_float("plr_sigma", 0.5, 10.0, log=True),
        "plr_act_name": trial.suggest_categorical("plr_act_name", ["relu", "prelu", "silu"]),
        "plr_lr_factor": trial.suggest_float("plr_lr_factor", 0.01, 0.5, log=True),

        "lr": trial.suggest_float("lr", 1e-4, 5e-2, log=True),
        "mom": trial.suggest_float("mom", 0.8, 0.99),
        "sq_mom": trial.suggest_float("sq_mom", 0.95, 0.999),
        "lr_sched": trial.suggest_categorical("lr_sched", ["constant", "flat_cos", "cos"]),
        "first_layer_lr_factor": trial.suggest_float("first_layer_lr_factor", 0.5, 2.0),
        "scale_lr_factor": trial.suggest_float("scale_lr_factor", 1.0, 20.0),
        "bias_lr_factor": trial.suggest_float("bias_lr_factor", 0.01, 1.0, log=True),
        "wd": trial.suggest_float("wd", 1e-5, 1e-1, log=True),
        "bias_wd_factor": trial.suggest_float("bias_wd_factor", 0.01, 1.0, log=True),

        "ls_eps": trial.suggest_float("ls_eps", 0.0, 0.2),
        "ls_eps_sched": trial.suggest_categorical("ls_eps_sched", ["constant", "cos"]),

        "tfms": ["median_center", "robust_scale", "smooth_clip"],

        "n_epochs": trial.suggest_int("n_epochs", 4, 20),
        "batch_size": trial.suggest_categorical("batch_size", [128, 256, 512]),
        "predict_batch_size": 10240,
        "verbosity": 0,

        "use_early_stopping": True,
        "early_stopping_additive_patience": 5,
        "early_stopping_multiplicative_patience": 1,

        "device": "cuda",
        "random_state": 42,
    }
    
    model = RealMLP_TD_Classifier(**config)

    model.fit(
        X_train_full,
        y_train,
        X_test_full,
        y_test,
        cat_col_names=list(c_cols),
    )

    # =========================================
    # Validation Prediction
    # =========================================

    y_pred = model.predict_proba(X_test_full)[:, 1]

    auc = roc_auc_score(y_test, y_pred)

    return auc


if RMLP_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=3,   # fewer random trials
        multivariate=True,
        group=True
    )
    
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="realmlp_auc_search"
    )
    
    # Add your default configuration as Trial 0
    study.enqueue_trial(CONFIG)
    
    # Start optimization
    study.optimize(
        objective,
        n_trials=25,
        show_progress_bar=True
    )
    
    print("\nBest AUC:")
    print(study.best_value)
    
    print("\nBest Parameters:")
    print(study.best_params)

CPU times: user 6 µs, sys: 0 ns, total: 6 µs
Wall time: 12.6 µs


## XGBoost - Optuna

In [18]:
def objective(trial):

    params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda",
    "n_estimators": trial.suggest_int("n_estimators", 500, 4000),
    "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.08, log=True),
    "max_depth": trial.suggest_int("max_depth", 4, 16),
    "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 20.0, log=True),
    "subsample": trial.suggest_float("subsample", 0.5, 1.0),
    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
    "gamma": trial.suggest_float("gamma", 1e-5, 10.0, log=True),
    "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
    "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
    "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
    "max_bin": trial.suggest_int("max_bin", 128, 512),
    "random_state": 42,
    "n_jobs": -1,
    }
    model = XGBClassifier(**params)
    model.fit(X_train_full, y_train)
    y_pred = model.predict_proba(X_test_full)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    print(f"Trial {trial.number} | ROC_AUC: {auc:.6f}")
    return auc


def best_trial_callback(study, trial):

    print("\n" + "="*60)
    print(f"Trial {trial.number} Completed")
    print(f"Trial ROC_AUC: {trial.value:.6f}")
    print(f"\nBest ROC_AUC So Far: {study.best_value:.6f}")
    print("\nBest Parameters So Far:")
    print(study.best_params)
    print("="*60)

if XGB_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=3,
        multivariate=True,
        group=True
    )
    
    study_xgb = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="xgboost_auc_search"
    )
    
    study_xgb.enqueue_trial(best_xgb_params)
    study_xgb.optimize(objective, n_trials=50, callbacks=[best_trial_callback], show_progress_bar=True)
    
    print("\nBest AUC:")
    print(study_xgb.best_value)
    
    print("\nBest Parameters:")
    print(study_xgb.best_params)

## LightGBM - Optuna

In [19]:
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 500, 4000),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.08, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 512),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 0, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 100.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
        "max_bin": trial.suggest_int("max_bin", 64, 255),
        "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.0),
        "verbosity": -1,
        "random_state": 42,
        "n_jobs": -1,
        "device_type": "gpu",
    }

    model = LGBMClassifier(**params)
    model.fit(X_train_full, y_train)
    y_pred = model.predict_proba(X_test_full)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    print(f"Trial {trial.number} | ROC_AUC: {auc:.6f}")
    return auc


def best_trial_callback(study, trial):

    print("\n" + "="*60)

    print(f"Trial {trial.number} Completed")
    print(f"Trial ROC_AUC: {trial.value:.6f}")

    print("\nBest ROC_AUC So Far:")
    print(f"{study.best_value:.6f}")

    print("\nBest Parameters So Far:")
    print(study.best_params)

    print("="*60)


if LGB_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=3,
        multivariate=True,
        group=True
    )
    
    study_lgb = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="lightgbm_auc_search"
    )
    
    study_lgb.enqueue_trial(lgb_best_params)
    study_lgb.optimize(objective, n_trials=50, callbacks=[best_trial_callback], show_progress_bar=True)
    
    print("\nBest AUC:")
    print(study_lgb.best_value)
    
    print("\nBest Parameters:")
    print(study_lgb.best_params)

## CatBoost - Optuna

In [20]:
def objective(trial):

    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "task_type": "GPU",
        "iterations": trial.suggest_int("iterations", 500, 4000),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.08, log=True),
        "depth": trial.suggest_int("depth", 5, 10),
        "bootstrap_type": trial.suggest_categorical(
            "bootstrap_type", ["Bernoulli", "Bayesian"]
        ),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 100.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-5, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 10.0),
        "border_count": trial.suggest_int("border_count", 64, 255),
        "metric_period": 10,
        "random_seed": 42,
        "verbose": 0,
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train_full, y_train)
    y_pred = model.predict_proba(X_test_full)[:, 1]
    auc = roc_auc_score(y_test, y_pred)
    print(f"Trial {trial.number} | ROC_AUC: {auc:.6f}")
    return auc


def best_trial_callback(study, trial):

    print("\n" + "="*60)
    print(f"Trial {trial.number} Completed")
    print(f"Trial ROC_AUC: {trial.value:.6f}")
    print(f"\nBest ROC_AUC So Far: {study.best_value:.6f}")
    print("\nBest Parameters So Far:")
    print(study.best_params)
    print("="*60)

if CAT_OPTUNA:
    sampler = optuna.samplers.TPESampler(
        seed=42,
        n_startup_trials=3,
        multivariate=True,
        group=True
    )
    
    study_cb = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        study_name="catboost_auc_search"
    )
    
    study_cb.enqueue_trial(cat_params)
    study_cb.optimize(objective, n_trials=50, callbacks=[best_trial_callback], show_progress_bar=True)
    
    print("\nBest AUC:")
    print(study_cb.best_value)
    
    print("\nBest Parameters:")
    print(study_cb.best_params)